# VGGT-SLAM GPU Reconstruction
Runtime → GPU → Run all → Upload images → Download

## 1. Check GPU

In [ ]:
import torch
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Install Dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq git libboost-all-dev cmake gcc g++ > /dev/null 2>&1
print('✅ Done')

## 3. Setup VGGT-SLAM

In [ ]:
import os
if not os.path.exists('VGGT-SLAM'):
    !git clone https://github.com/MIT-SPARK/VGGT-SLAM.git
    print('✅ Cloned')
else:
    print('✅ Exists')

In [ ]:
%%bash
cd VGGT-SLAM
chmod +x setup.sh
./setup.sh

## 4. Upload Images

In [ ]:
from google.colab import files
import zipfile, os, shutil

if os.path.exists('input_images'):
    shutil.rmtree('input_images')
os.makedirs('input_images')

print('📤 Upload images_for_vggt_slam.zip')
uploaded = files.upload()

for f in uploaded:
    if f.endswith('.zip'):
        print(f'Extracting {f}...')
        with zipfile.ZipFile(f) as z:
            z.extractall('input_images')
        os.remove(f)

imgs = [f for f in os.listdir('input_images') if f.lower().endswith(('.jpg','.png'))]
print(f'✅ {len(imgs)} images')

## 5. Run VGGT-SLAM (30-40 min)

In [ ]:
%%bash
cd VGGT-SLAM
mkdir -p ../vggt_logs
echo '🚀 Starting...'
python3 main.py --image_folder ../input_images --max_loops 3 --conf_threshold 25.0 --log_results --log_path ../vggt_logs/poses.txt
echo '✅ Done'

## 6. Export Point Cloud

In [ ]:
import glob, os, shutil
import numpy as np

print('📊 Checking output...\n')

# Check vggt_logs
if not os.path.exists('vggt_logs'):
    print('❌ vggt_logs does not exist!')
    raise Exception('VGGT-SLAM did not run')

print('Files in vggt_logs:')
for item in os.listdir('vggt_logs'):
    if os.path.isdir(os.path.join('vggt_logs', item)):
        print(f'  {item}/ (folder)')
    else:
        print(f'  {item}')
print()

# Find point clouds - VGGT-SLAM saves as .npz files
npz_files = glob.glob('vggt_logs/poses_logs/*.npz')

print(f'Found {len(npz_files)} NPZ files\n')

if not npz_files:
    print('❌ NO POINT CLOUDS!')
    raise Exception('No NPZ files found in poses_logs/')

os.makedirs('vggt_output', exist_ok=True)

# Process NPZ files
print(f'Processing {len(npz_files)} NPZ files...\n')
all_pts, all_cols = [], []

for npz_path in sorted(npz_files):
    fname = os.path.basename(npz_path)
    print(f'  Reading {fname}...')
    data = np.load(npz_path)
    
    # VGGT-SLAM uses 'pointcloud' and 'mask' keys
    if 'pointcloud' in data:
        pc = data['pointcloud']  # Shape: (H, W, 3) - organized point cloud
        mask = data.get('mask', None)  # Optional mask for valid points
        
        if len(all_pts) == 0:
            print(f'    Shape: {pc.shape}')
        
        # Reshape from (H, W, 3) to (H*W, 3)
        if pc.ndim == 3 and pc.shape[2] == 3:
            h, w = pc.shape[:2]
            pts = pc.reshape(-1, 3)  # Flatten to (N, 3)
            
            # Apply mask to filter valid points
            if mask is not None:
                mask_flat = mask.reshape(-1)
                pts = pts[mask_flat]
            
            # Filter out invalid points (NaN, inf, or zero)
            valid = np.isfinite(pts).all(axis=1) & (np.abs(pts).sum(axis=1) > 0.01)
            pts = pts[valid]
            
            if len(pts) > 0:
                all_pts.append(pts)
                # Default white color for now (no color data in NPZ)
                all_cols.append(np.full((len(pts), 3), 200, dtype=np.uint8))
                print(f'    ✓ {len(pts):,} valid points')
            else:
                print(f'    ⚠ No valid points after filtering')

if not all_pts:
    print('\n❌ No valid point data found in NPZ files!')
    raise Exception('Could not extract points from NPZ files')

# Merge all points
pts = np.vstack(all_pts)
cols = np.vstack(all_cols)

print(f'\n✅ Merged {len(pts):,} points\n')

# Write PLY
ply = 'vggt_output/point_cloud.ply'
with open(ply, 'w') as f:
    f.write('ply\n')
    f.write('format ascii 1.0\n')
    f.write(f'element vertex {len(pts)}\n')
    f.write('property float x\n')
    f.write('property float y\n')
    f.write('property float z\n')
    f.write('property uchar red\n')
    f.write('property uchar green\n')
    f.write('property uchar blue\n')
    f.write('end_header\n')
    for (x,y,z), (r,g,b) in zip(pts, cols):
        f.write(f'{x} {y} {z} {int(r)} {int(g)} {int(b)}\n')

print(f'✅ point_cloud.ply ({os.path.getsize(ply)/1024/1024:.1f} MB)')

# Copy poses
if os.path.exists('vggt_logs/poses.txt'):
    shutil.copy('vggt_logs/poses.txt', 'vggt_output/camera_poses.txt')
    print('✅ camera_poses.txt')

print('\n✅ Export complete!')

## 7. Download

In [ ]:
import zipfile, os
from google.colab import files

print('📦 Creating ZIP...\n')

if not os.path.exists('vggt_output'):
    print('❌ vggt_output does not exist!')
    raise Exception('No output')

output_files = []
for root, dirs, fnames in os.walk('vggt_output'):
    for f in fnames:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp)
        output_files.append((fp, f))
        print(f'  {f} ({sz/1024:.1f} KB)')

if not output_files:
    print('\n❌ vggt_output is EMPTY!')
    raise Exception('No files to zip')

with zipfile.ZipFile('vggt_slam_output.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for fp, fn in output_files:
        z.write(fp, fn)

sz = os.path.getsize('vggt_slam_output.zip')/1024/1024
print(f'\n✅ ZIP: {sz:.1f} MB')
print('📥 Downloading...')
files.download('vggt_slam_output.zip')
print('✅ DONE')